In [5]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [6]:
# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

elevation = dataset.select('DEM')

elev_projection = elevation.first().projection()

In [ ]:
pa_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(panama_geom)

pa_raster = ee.Image().byte().paint(pa_fc, 1)

dist_pa = (
    pa_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_pa")
)

In [31]:
dist_pa.projection()

In [30]:
Map = geemap.Map()
Map.centerObject(panama_geom, 7)

Map.addLayer(pa_fc, {}, 'Protected Areas')
# Map.addLayer(pa_raster, {}, 'Protected Areas Rasterized')
Map.addLayer(dist_pa, {min: 0, max: 1000, 'palette': ['blue', 'green', 'yellow']}, 'Dist to PAs')

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…

In [33]:
# export this as an image asset with the elev_projection and scale.
task = ee.batch.Export.image.toAsset(
    image = dist_pa,
    description = 'distance_protected_areas',
    assetId = "projects/deforestation-panama/assets/distance_loss",
    region = panama_geom,
    scale = elev_projection.nominalScale(),
    crs = elev_projection.crs(),
    crsTransform = elev_projection.getInfo()['transform'],
    maxPixels=1e12
)

task.start()

In [ ]:
# export this as an image asset with the elev_projection and scale.
task = ee.batch.Export.image.toAsset(
    image = distance_loss,
    description = 'distance_loss',
    assetId = "projects/deforestation-panama/assets/distance_loss",
    region = panama_geom,
    scale = elev_projection.nominalScale(),
    crs = elev_projection.crs(),
    crsTransform = elev_projection.getInfo()['transform'],
    maxPixels=1e12
)

task.start()